# Example of TwoTower training/inference

The standard [TwoTower example](15_twotower_example.ipynb) shows the simplest setup with `item_id` and `TwoTower.from_params`. This advanced example adds the parts needed for a more flexible and efficient training setup:

- usage of additional item features, including their preprocessing and encoding;
- an explicit `ParquetModule` transform pipeline instead of a default transform factory in order to use negative sampling.
- independent negative pools for logical groups packed into one larger physical batch;
- a `TwoTower` model created from individual building blocks for non-standard configuration;
- target-aware trimming and length-bucketed query encoding to reduce work on left padding;
- optional chunked item-cache construction to reduce peak GPU memory use.

This example uses `ParquetModule` to load and process data.

Set `ARROW_NUM_THREADS` and `ARROW_IO_THREADS` to the number of available CPU cores. Set the OpenMP, MKL, and PyTorch thread counts to `1` so that they do not compete with Arrow workers.


In [ ]:
%env OMP_NUM_THREADS=1
%env MKL_NUM_THREADS=1
%env ARROW_NUM_THREADS=8
%env ARROW_IO_THREADS=8

env: OMP_NUM_THREADS=1
env: MKL_NUM_THREADS=1
env: ARROW_NUM_THREADS=8
env: ARROW_IO_THREADS=8


In [ ]:
import lightning as L
import pandas as pd
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

L.seed_everything(42)

Seed set to 42


42

## Preparing data

This example uses MovieLens-1M from `examples/data`. The model uses `item_id` from the interactions and two item features: `movie_title` and `genre`.

The current `TwoTower` implementation supports item and interaction features. It does not use user features as model inputs.


In [ ]:
user_features = pd.read_csv(
    "./data/ml1m_users.dat",
    sep="\t",
    names=["user_id", "gender", "age", "occupation", "zip_code"],
)[["user_id"]]
item_features = pd.read_csv(
    "./data/ml1m_items.dat",
    sep="\t",
    names=["item_id", "movie_title", "genre"],
)
interactions = pd.read_csv(
    "./data/ml1m_ratings.dat",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
).drop(columns=["rating"])

In [ ]:
interactions["timestamp"] = interactions["timestamp"].astype("int64")
interactions = interactions.sort_values(by="timestamp")
interactions["timestamp"] = interactions.groupby("user_id").cumcount()
interactions.head(5)

,user_id,item_id,timestamp
1000138,6040,858,0
999873,6040,593,1
1000153,6040,2384,2
1000192,6040,2019,3
1000007,6040,1961,4


### Encode categorical data

Use `LabelEncoder` to encode `user_id` and `item_id`. Fit the rules on the user and item catalogs so that validation and test data do not contain unknown IDs.


In [ ]:
from replay.preprocessing import LabelEncoder, LabelEncodingRule

encoder = LabelEncoder(
    [
        LabelEncodingRule("user_id"),
        LabelEncodingRule("item_id"),
    ]
)
encoder.rules[0].fit(user_features)
encoder.rules[1].fit(item_features)

encoded_interactions = encoder.transform(interactions)
item_features_encoded = encoder.rules[1].transform(item_features)
item_features_encoded.head(5)

,movie_title,genre,item_id
0,Toy Story (1995),Animation|Children's|Comedy,0
1,Jumanji (1995),Adventure|Children's|Fantasy,1
2,Grumpier Old Men (1995),Comedy|Romance,2
3,Waiting to Exhale (1995),Comedy|Drama,3
4,Father of the Bride Part II (1995),Comedy,4


#### Prepare item features

RePlay transformer models support four feature types:

- `CATEGORICAL`: one categorical value per object, such as an item ID;
- `CATEGORICAL_LIST`: several categorical values per object, such as movie genres;
- `NUMERICAL`: one numerical value per object;
- `NUMERICAL_LIST`: several numerical values per object, such as a text embedding.

Set the type with `feature_type` in `TensorSchema`.


A movie can have several genres, so `genre` is a `CATEGORICAL_LIST` feature. The source file stores genres in one string. Split that string into a list first.


In [ ]:
item_features_encoded["genre"] = item_features_encoded["genre"].str.split("|", regex=False)
item_features_encoded.head(5)

,movie_title,genre,item_id
0,Toy Story (1995),"[Animation, Children's, Comedy]",0
1,Jumanji (1995),"[Adventure, Children's, Fantasy]",1
2,Grumpier Old Men (1995),"[Comedy, Romance]",2
3,Waiting to Exhale (1995),"[Comedy, Drama]",3
4,Father of the Bride Part II (1995),[Comedy],4


Encode the genre lists with `SequenceEncodingRule`.


In [ ]:
from replay.preprocessing import SequenceEncodingRule

genre_encoder = LabelEncoder([SequenceEncodingRule("genre")])
item_features_encoded = genre_encoder.fit_transform(item_features_encoded)
item_features_encoded.head(5)

,movie_title,item_id,genre
0,Toy Story (1995),0,"[2, 3, 4]"
1,Jumanji (1995),1,"[1, 3, 8]"
2,Grumpier Old Men (1995),2,"[4, 13]"
3,Waiting to Exhale (1995),3,"[4, 7]"
4,Father of the Bride Part II (1995),4,[4]


Save the longest genre list and the number of unique genres. These values are needed later in the metadata for `ParquetModule` and `ItemTower`.


In [ ]:
MAX_LEN_GENRE = int(item_features_encoded["genre"].apply(len).max())
NUM_UNIQUE_GENRE_VALUES = int(len(genre_encoder.mapping["genre"]))

`movie_title` is a text feature. There are two common ways to encode it:

- Use `CountVectorizer` or a similar method. The result is a `CATEGORICAL_LIST` feature.
- Use a pretrained text model. The result is a `NUMERICAL_LIST` feature. If its size differs from the model embedding size, RePlay adds a linear projection.

This example uses a model from `sentence-transformers`.


In [ ]:
%pip install sentence-transformers -q


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

text_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = item_features_encoded["movie_title"].fillna("").tolist()
embeddings = text_encoder.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)
item_features_encoded["movie_title"] = [value.astype(np.float32) for value in embeddings]
TITLE_EMB_DIM = int(text_encoder.get_embedding_dimension())

item_features_encoded.head(5)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,movie_title,item_id,genre
0,"[-0.082835145, 0.053030428, 0.053576548, -0.02...",0,"[2, 3, 4]"
1,"[-0.10529309, 0.15084095, -0.02639786, -0.0655...",1,"[1, 3, 8]"
2,"[-0.09878677, 0.017649995, -0.05274402, -0.038...",2,"[4, 13]"
3,"[-0.08723061, 0.03661162, -0.021703225, -0.012...",3,"[4, 7]"
4,"[-0.0692064, 0.03875223, 0.01473796, 0.0121407...",4,[4]


### Split interactions into the train, validation and test datasets using LastNSplitter

Use the last-one-out strategy. Cold users and items are removed for simplicity.


In [11]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True,
)

test_events, test_gt = splitter.split(encoded_interactions)
validation_events, validation_gt = splitter.split(test_events)
train_events = validation_events


### Dataset preprocessing ("baking")

Join the encoded item features to each interaction split. Then use `groupby_sequences` to group each user's events into a sequence sorted by time.


In [12]:
train_events = train_events.merge(item_features_encoded, on="item_id", how="inner")
validation_events = validation_events.merge(item_features_encoded, on="item_id", how="inner")
test_events = test_events.merge(item_features_encoded, on="item_id", how="inner")

In [ ]:
from replay.data.nn.utils import groupby_sequences


def bake_data(data: pd.DataFrame):
    return groupby_sequences(events=data, groupby_col="user_id", sort_col="timestamp")


train_events = bake_data(train_events)
validation_events = bake_data(validation_events)
validation_gt = bake_data(validation_gt)
test_events = bake_data(test_events)
test_gt = bake_data(test_gt)

train_events.head(3)

,user_id,timestamp,item_id,movie_title,genre
0,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[3117, 1009, 1250, 1672, 2271, 1768, 3339, 118...","[[-0.11641536, -0.063095465, 0.0054702833, -0....","[[7], [2, 3, 11], [4, 14], [7, 13], [13], [7],..."
1,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1180, 1199, 1192, 2648, 1273, 2874, 1207, 117...","[[-0.027989453, 0.07571321, -0.06741784, 0.025...","[[0, 1], [7, 16], [0, 1, 13, 14, 16], [4, 10],..."
2,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[589, 2789, 3465, 1899, 1892, 1407, 1246, 1358...","[[-0.07848771, -0.00096751953, -0.019314129, 0...","[[7, 15], [4, 7], [4], [4, 7], [7], [0, 4], [1..."


Join each event dataset with its ground truth by `user_id`. This keeps only users that are present in both datasets and provides `ground_truth` for validation.


In [ ]:
def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})
    return events_df.merge(gt_to_join, on="user_id", how="inner")


validation_events = add_gt_to_events(validation_events, validation_gt)
test_events = add_gt_to_events(test_events, test_gt)

The two towers read their data separately. The query tower reads interaction sequences one batch at a time. The item tower loads the complete encoded item catalog and keeps it in memory.


In [15]:
from pathlib import Path

data_dir = Path("temp/data/")
data_dir.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = data_dir / "train.parquet"
VAL_PATH = data_dir / "val.parquet"
TEST_PATH = data_dir / "test.parquet"
PATH_ENCODED_FEATURES = data_dir / "item_features_encoded.parquet"

ENCODER_PATH = data_dir / "encoder"
GENRE_ENCODER_PATH = data_dir / "genre_encoder"

In [ ]:
item_feature_names = ["item_id", "genre", "movie_title"]
item_features_encoded[item_feature_names].to_parquet(PATH_ENCODED_FEATURES)

event_feature_names = item_feature_names + ["user_id"]
train_events[event_feature_names].to_parquet(TRAIN_PATH)
validation_events[event_feature_names + ["ground_truth"]].to_parquet(VAL_PATH)
test_events[event_feature_names + ["ground_truth"]].to_parquet(TEST_PATH)

encoder.save(ENCODER_PATH)
genre_encoder.save(GENRE_ENCODER_PATH)

# Prepare to model training

### Create the tensor schema

`TensorSchema` maps source columns to tensors used by the model. Categorical features need `cardinality` and `padding_value`. Numerical lists need `tensor_dim`. The `feature_sources` field tells `TwoTower` where each item feature comes from. `user_id` is not part of the schema because it is not a model input.


In [17]:
from replay.data import FeatureHint, FeatureSource, FeatureType
from replay.data.nn import TensorFeatureInfo, TensorFeatureSource, TensorSchema

EMBEDDING_DIM = 192
NUM_UNIQUE_ITEMS = len(encoder.mapping["item_id"])

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            "item_id",
            is_seq=True,
            feature_type=FeatureType.CATEGORICAL,
            embedding_dim=EMBEDDING_DIM,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            feature_hint=FeatureHint.ITEM_ID,
            feature_sources=[TensorFeatureSource(FeatureSource.INTERACTIONS, "item_id")],
        ),
        TensorFeatureInfo(
            "genre",
            is_seq=True,
            feature_type=FeatureType.CATEGORICAL_LIST,
            embedding_dim=EMBEDDING_DIM,
            padding_value=NUM_UNIQUE_GENRE_VALUES,
            cardinality=NUM_UNIQUE_GENRE_VALUES,
            feature_sources=[TensorFeatureSource(FeatureSource.ITEM_FEATURES, "genre")],
        ),
        TensorFeatureInfo(
            "movie_title",
            is_seq=True,
            feature_type=FeatureType.NUMERICAL_LIST,
            embedding_dim=EMBEDDING_DIM,
            tensor_dim=TITLE_EMB_DIM,
            feature_sources=[TensorFeatureSource(FeatureSource.ITEM_FEATURES, "movie_title")],
        ),
    ]
)


### Configure ParquetModule and transformation pipelines

`ParquetModule` reads every data split in batches and applies a split-specific transform pipeline before the model forward pass.

The training pipeline uses two optimization pairs:

1. `GroupedUniformNegativeSamplingTransform` creates an independent negative pool for each logical group. Its `group_size` must match `logical_batch_size` in `GroupedCESampled`.
2. `TargetAwareAdaptiveTrimTransform` removes left padding that is not used by inputs or targets. `LengthBucketedQueryEncoder` then reduces the sequence length inside groups of similarly sized sequences.

The transforms are listed explicitly so that their order is easy to see. `NextTokenTransform` uses `shift=1`, so the training metadata must read `MAX_SEQ_LEN + 1` items.


In [ ]:
from replay.nn.transform import (
    GroupedUniformNegativeSamplingTransform,
    GroupTransform,
    NextTokenTransform,
    RenameTransform,
    TargetAwareAdaptiveTrimTransform,
    UnsqueezeTransform,
)

LOGICAL_BATCH_SIZE = 32
NUM_NEGATIVE_SAMPLES = min(1_000, NUM_UNIQUE_ITEMS - 1)
ITEM_ID_NAME = tensor_schema.item_id_feature_name

train_transforms = [
    NextTokenTransform(label_name=ITEM_ID_NAME, shift=1),
    RenameTransform(
        {
            f"{ITEM_ID_NAME}_mask": "padding_mask",
            "positive_labels_mask": "target_padding_mask",
        }
    ),
    # Both masks must still be two-dimensional at this point.
    TargetAwareAdaptiveTrimTransform(
        feature_names=[*tensor_schema.sequential_features.names, "positive_labels"],
    ),
    UnsqueezeTransform("target_padding_mask", -1),
    UnsqueezeTransform("positive_labels", -1),
    GroupTransform({"feature_tensors": tensor_schema.names}),
    GroupedUniformNegativeSamplingTransform(
        cardinality=NUM_UNIQUE_ITEMS,
        num_negative_samples=NUM_NEGATIVE_SAMPLES,
        group_size=LOGICAL_BATCH_SIZE,
    ),
]


def make_inference_transforms():
    return [
        RenameTransform({f"{ITEM_ID_NAME}_mask": "padding_mask"}),
        GroupTransform({"feature_tensors": tensor_schema.names}),
    ]


transforms = {
    "train": train_transforms,
    "validate": make_inference_transforms(),
    "test": make_inference_transforms(),
    "predict": make_inference_transforms(),
}

Metadata describes the shape and padding value of every input column. For scalar sequence features, `shape` is the sequence length. For list sequence features, use `[sequence_length, feature_size]`. Use plain Python `int` values in metadata.


In [ ]:
def create_metadata(sequence_length: int, ground_truth_length: int | None = None):
    metadata = {
        "item_id": {"shape": sequence_length, "padding": tensor_schema["item_id"].padding_value},
        "genre": {
            "shape": [sequence_length, MAX_LEN_GENRE],
            "padding": tensor_schema["genre"].padding_value,
        },
        "movie_title": {
            "shape": [sequence_length, TITLE_EMB_DIM],
            "padding": tensor_schema["movie_title"].padding_value,
        },
    }
    if ground_truth_length is not None:
        metadata["ground_truth"] = {"shape": ground_truth_length, "padding": -1}
    return metadata


MAX_SEQ_LEN = 50
validation_gt_length = int(validation_events["ground_truth"].apply(len).max())

train_metadata = {
    "train": create_metadata(sequence_length=MAX_SEQ_LEN + 1),
    "validate": create_metadata(
        sequence_length=MAX_SEQ_LEN,
        ground_truth_length=validation_gt_length,
    ),
}

In [ ]:
from replay.data.nn import ParquetModule

# Several logical groups are packed into one physical data-loader batch.
BATCH_SIZE = 4 * LOGICAL_BATCH_SIZE

parquet_module = ParquetModule(
    train_path=TRAIN_PATH,
    validate_path=VAL_PATH,
    batch_size=BATCH_SIZE,
    metadata=train_metadata,
    transforms=transforms,
)

/tmp/ipykernel_28319/2276288214.py:6: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


## Train model

### Create the TwoTower model instance and run the training stage using lightning

RePlay's `TwoTower` uses a modular, block-based design. This example builds the model from individual components because the optimized training path needs a custom query encoder and loss.

#### Advanced Configuration


`FeaturesReader` loads the complete item catalog. Its metadata follows the same basic rules as `ParquetModule` metadata, but catalog rows are not sequences. Use an empty dictionary for scalar features and specify `shape` only for list features.

During validation or inference, `TwoTower` builds an item cache so that item embeddings are computed only once. The `item_cache_batch_size` parameter is optional. If it is not set, the entire catalog is encoded in one pass and the full item cache is kept in memory. Set this parameter to build the cache in smaller chunks when encoding the entire catalog at once would exceed GPU memory. Chunking reduces peak memory use during cache construction, but makes this one-time operation slower. It does not change the size of the completed cache. This example sets `ITEM_CACHE_BATCH_SIZE` below and passes it to `TwoTowerBody`.


In [ ]:
feature_reader_metadata = {
    "item_id": {},
    "genre": {"shape": MAX_LEN_GENRE, "padding": tensor_schema["genre"].padding_value},
    "movie_title": {
        "shape": TITLE_EMB_DIM,
        "padding": tensor_schema["movie_title"].padding_value,
    },
}

In [ ]:
from replay.nn.agg import SumAggregator
from replay.nn.embedding import SequenceEmbedding
from replay.nn.ffn import SwiGLUEncoder
from replay.nn.loss import GroupedCESampled
from replay.nn.mask import DefaultAttentionMask
from replay.nn.sequential import LengthBucketedQueryEncoder, PositionAwareAggregator, SasRecTransformerLayer, TwoTower
from replay.nn.sequential.twotower import FeaturesReader, TwoTowerBody

NUM_HEADS = 2
NUM_BLOCKS = 2
DROPOUT = 0.2
LENGTH_BUCKET_SIZE = 32
ITEM_CACHE_BATCH_SIZE = 512


def build_twotower() -> TwoTower:
    common_aggregator = SumAggregator(embedding_dim=EMBEDDING_DIM)
    query_encoder = LengthBucketedQueryEncoder(
        encoder=SasRecTransformerLayer(
            embedding_dim=EMBEDDING_DIM,
            num_heads=NUM_HEADS,
            num_blocks=NUM_BLOCKS,
            dropout=DROPOUT,
            activation="relu",
        ),
        bucket_size=LENGTH_BUCKET_SIZE,
        sequence_shift=1,
    )
    body = TwoTowerBody(
        schema=tensor_schema,
        embedder=SequenceEmbedding(schema=tensor_schema),
        attn_mask_builder=DefaultAttentionMask(
            reference_feature_name=tensor_schema.item_id_feature_name,
            num_heads=NUM_HEADS,
        ),
        query_tower_feature_names=tensor_schema.names,
        query_embedding_aggregator=PositionAwareAggregator(
            embedding_aggregator=common_aggregator,
            max_sequence_length=MAX_SEQ_LEN,
            dropout=DROPOUT,
        ),
        item_embedding_aggregator=common_aggregator,
        query_encoder=query_encoder,
        query_tower_output_normalization=torch.nn.LayerNorm(EMBEDDING_DIM),
        item_encoder=SwiGLUEncoder(
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=2 * EMBEDDING_DIM,
        ),
        item_features_reader=FeaturesReader(
            schema=tensor_schema,
            metadata=feature_reader_metadata,
            path=PATH_ENCODED_FEATURES,
        ),
        item_cache_batch_size=ITEM_CACHE_BATCH_SIZE,
    )
    loss = GroupedCESampled(
        logical_batch_size=LOGICAL_BATCH_SIZE,
        cardinality=NUM_UNIQUE_ITEMS,
    )
    return TwoTower(body=body, loss=loss)


twotower = build_twotower()

A universal PyTorch Lightning module is provided that can work with any NN model.


In [ ]:
from replay.nn.lightning import LightningModule

model = LightningModule(twotower)

To facilitate training, add the following callbacks:

1. `ModelCheckpoint` saves the model with the best Recall value.
2. `ComputeMetricsCallback` calculates validation metrics after every epoch.


In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from replay.nn.lightning.callback import ComputeMetricsCallback

checkpoint_callback = ModelCheckpoint(
    dirpath="twotower/checkpoints/",
    save_top_k=1,
    verbose=True,
    monitor="recall@10",
    mode="max",
)

validation_metrics_callback = ComputeMetricsCallback(
    metrics=["map", "ndcg", "recall"],
    ks=[1, 5, 10, 20],
    item_count=NUM_UNIQUE_ITEMS,
    verbose=False,
)

csv_logger = CSVLogger(save_dir="twotower/.logs/train", name="TwoTower-advanced-example")

trainer = L.Trainer(
    max_epochs=5,
    callbacks=[checkpoint_callback, validation_metrics_callback],
    logger=csv_logger,
)

trainer.fit(model, datamodule=parquet_module)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ TwoTower │  1.7 M │ train │     0 │
└───┴───────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.7 M                                                                                                
Total estimated model params size (MB): 6.897                                                                      
Modules in train mode: 58                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/replay/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

Epoch 0, global step 48: 'recall@10' reached 0.02815 (best 0.02815), saving model to '/home/evtsinovnik/RePlay/examples/twotower/checkpoints/epoch=0-step=48.ckpt' as top 1


Epoch 1, global step 96: 'recall@10' reached 0.04487 (best 0.04487), saving model to '/home/evtsinovnik/RePlay/examples/twotower/checkpoints/epoch=1-step=96.ckpt' as top 1


Epoch 2, global step 144: 'recall@10' reached 0.05712 (best 0.05712), saving model to '/home/evtsinovnik/RePlay/examples/twotower/checkpoints/epoch=2-step=144.ckpt' as top 1


Epoch 3, global step 192: 'recall@10' reached 0.07219 (best 0.07219), saving model to '/home/evtsinovnik/RePlay/examples/twotower/checkpoints/epoch=3-step=192.ckpt' as top 1


Epoch 4, global step 240: 'recall@10' reached 0.09123 (best 0.09123), saving model to '/home/evtsinovnik/RePlay/examples/twotower/checkpoints/epoch=4-step=240.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=5` reached.


Now get the best checkpoint path and print the validation metrics from its epoch.


In [ ]:
best_model_path = checkpoint_callback.best_model_path
best_epoch = int(best_model_path.split("epoch=")[1].split("-")[0])
best_epoch_validation_metrics = validation_metrics_callback.get_metrics()[best_epoch]

best_model_path, best_epoch_validation_metrics

('/home/RePlay/examples/twotower/checkpoints/epoch=4-step=240.ckpt',
 {'recall@1': 0.00612582778558135,
  'ndcg@1': 0.00612582778558135,
  'map@1': 0.00612582778558135,
  'recall@5': 0.0466887466609478,
  'ndcg@5': 0.02607857622206211,
  'map@5': 0.019359823316335678,
  'recall@10': 0.09122516959905624,
  'ndcg@10': 0.04027450829744339,
  'map@10': 0.025097036734223366,
  'recall@20': 0.15695364773273468,
  'ndcg@20': 0.056797560304403305,
  'map@20': 0.02958284504711628})

## Inference

Load the best checkpoint with `LightningModule.load_from_checkpoint`. The new model instance must have the same architecture as the model used for training.


In [ ]:
twotower = build_twotower()

best_model = LightningModule.load_from_checkpoint(
    best_model_path,
    model=twotower,
    weights_only=False,
)
best_model.eval();

Configure `ParquetModule` for inference. Add `user_id` to the metadata so that predictions can be mapped back to users. Use an empty dictionary for this scalar column.


In [27]:
inference_metadata = {
    "predict": create_metadata(sequence_length=MAX_SEQ_LEN) | {"user_id": {}},
}

parquet_module = ParquetModule(
    predict_path=TEST_PATH,
    batch_size=BATCH_SIZE,
    metadata=inference_metadata,
    transforms=transforms,
)


/tmp/ipykernel_28319/567310089.py:5: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


During inference, use `PandasTopItemsCallback` to collect the highest-scoring items in a Pandas DataFrame.

RePlay also provides `PolarsTopItemsCallback`, `SparkTopItemsCallback`, and `TorchTopItemsCallback` for other output formats.


In [28]:
from replay.nn.lightning.callback import PandasTopItemsCallback

csv_logger = CSVLogger(save_dir="twotower/.logs/test", name="TwoTower-advanced-example")

TOPK = [1, 5, 10, 20]
pandas_prediction_callback = PandasTopItemsCallback(
    top_k=max(TOPK),
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(
    callbacks=[pandas_prediction_callback],
    logger=csv_logger,
    inference_mode=True,
)
trainer.predict(best_model, datamodule=parquet_module, return_predictions=False)

pandas_res = pandas_prediction_callback.get_result()


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Output()

/home/replay/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/replay/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


In [29]:
pandas_res


,user_id,item_id,score
0,0,1838,71.835953
0,0,2618,71.823761
0,0,1949,71.715714
0,0,360,71.706978
0,0,591,71.675751
...,...,...,...
6037,6039,1229,83.418449
6037,6039,1176,83.391838
6037,6039,2789,83.383629
6037,6039,585,83.374481


### Calculating metrics

`test_gt` is already encoded, so it can be used directly to calculate metrics.


In [ ]:
from replay.metrics import MAP, OfflineMetrics, Precision, Recall
from replay.metrics.torch_metrics_builder import metrics_to_df

In [ ]:
result_metrics = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), MAP(TOPK)],
    query_column="user_id",
    rating_column="score",
)(pandas_res, test_gt.explode("item_id"))

In [ ]:
metrics_to_df(result_metrics)

k,1,5,10,20
MAP,0.009109,0.021381,0.026661,0.031049
Precision,0.009109,0.009738,0.009026,0.007726
Recall,0.009109,0.048692,0.090262,0.154521


Call `inverse_transform` on the fitted encoder to restore the original user and item IDs.


In [ ]:
encoder = LabelEncoder.load(ENCODER_PATH)
encoder.inverse_transform(pandas_res)

,user_id,item_id,score
0,1,1907,71.835953
0,1,2687,71.823761
0,1,2018,71.715714
0,1,364,71.706978
0,1,595,71.675751
...,...,...,...
6037,6040,1249,83.418449
6037,6040,1193,83.391838
6037,6040,2858,83.383629
6037,6040,589,83.374481
